In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling
!pip install rapidfuzz

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import unicodedata
import re
import numpy as np
import pandas as pd

In [ ]:
def nettoyer_html(texte):
    if pd.isna(texte):
        return texte

    soup = BeautifulSoup(texte, "html.parser")

    # transforme les <br> en retours à la ligne
    for br in soup.find_all("br"):
        br.replace_with("\n")

    texte = soup.get_text(separator=" ", strip=True)

    return texte

In [ ]:
#Afficher tout le df
pd.set_option('display.max_rows', None)

In [ ]:
#lire le fichier excel CONTACT ODOO
df_ticket_Odoo = pd.read_excel('/content/Ticket Helpdesk (helpdesk.ticket) (18).xlsx');

In [ ]:
df_ticket_Odoo2 = df_ticket_Odoo.dropna(subset=['Messages/Contenus', 'ID'], how='all').reset_index(drop=True)

In [ ]:
# On travaille sur une copie
df = df_ticket_Odoo.copy().reset_index(drop=True)

i = 0
while i < len(df):

    # Si la ligne est un ticket (ID non nul)
    if pd.notna(df.loc[i, 'ID']):

        j = i + 1

        # Tant que les lignes suivantes n'ont pas d'ID
        while j < len(df) and pd.isna(df.loc[j, 'ID']):

            # Récupère le texte
            texte = df.loc[j, 'Messages/Contenus']

            if pd.notna(texte):
                if pd.isna(df.loc[i, 'Messages/Contenus']):
                    df.loc[i, 'Messages/Contenus'] = texte
                else:
                    df.loc[i, 'Messages/Contenus'] += "\n" + texte

            j += 1

    i += 1

# On garde uniquement les vraies lignes de tickets
df = df[df['ID'].notna()].reset_index(drop=True)

In [ ]:
df["ID"] = df['ID'].astype(int)

df['Équipe Helpdesk'] = df['Équipe Helpdesk'].replace({
    "Assistance infogerance": "Infogerance",
    "Assistance Telecom": "Telecom",
    "Assistance Réseau": "Réseau"
})

df['Priorité'] = df['Priorité'].replace({
    "Tous": "3 - Non urgent 🟢",
    "Urgent": "1 - Majeur 🔴",
    "Haute priorité": "2 - Mineur 🟠",
    "Priorité basse": "3 - Non urgent 🟢"
})

df["Étape"] = df['Étape'].replace({
    "En cours": "Prise en compte",
    "Programmation du ticket": "Intervention programmée"
})

df['Email Client'] = (
    df['Email Client']
    .str.extract(r'<([^>]+)>', expand=False)
    .fillna(df['Email Client'])
)

df['Email cc'] = (
    df['Email cc']
    .str.findall(r'<([^>]+)>')
    .str.join('; ')
)

df["Projet"] = df['Projet'].replace({
    "Emails entrants": np.nan
})

df['Description complète'] = (
    df['Assigné à'].fillna('') +
    '\n' +
    df['Description'].fillna('')
).str.strip()

In [ ]:
df.columns

In [ ]:
df["Messages/Contenus"] = df["Messages/Contenus"].apply(nettoyer_html)

In [ ]:
df['Description complète'] = (
    df['Assigné à'].fillna('') +
    '\n' +
    df['Description'].fillna('')
).str.strip()

In [ ]:
df = df[['ID', 'Nom affiché', 'Assigné à', 'Créé le', 'Équipe Helpdesk',
       'Description', 'Priorité', 'Étape', 'Email cc', 'Email Client',
       'Projet', 'Messages/Contenus']]

In [ ]:
df.to_excel("tickets_odoo_nettoyes.xlsx", index=False)

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df['Étape'].value_counts()